In [1]:
using Distributions
using Serialization
using CSV
using Distributed
using ProgressMeter
include("gen_crp_softmax.jl")
# Set random seed for reproducibility

Random.seed!(1777)
n_subs=5


SystemError: SystemError: opening file "c:\\Users\\zyfxl\\OneDrive - UC Irvine\\UCI\\CCNL Lab\\Foraging Task Overview\\foraging_MQLP-main\\foraging_MQLP-main\\model\\model_parameter_recovery_clean_version\\generate sim data with free alpha range\\gen_crp_softmax.jl": No such file or directory

### Simulate subjects of a wide range of parameters (Main Simulation)

In [ ]:
using Distributions
using CSV
using DataFrames
using Random
using ProgressMeter

include("gen_crp_softmax.jl")


n_subs  = 1
rngseed = 1777

γbase_dist = Uniform(-10, 10)
γcoef_dist = Uniform(-3, 3)
# beta_dist  = Uniform(0, 5)
# eps_dist   = Uniform(0, 0.1)   # ← new
alpha_dist  = Uniform(0,10)

Random.seed!(rngseed)

all_data = DataFrame()
p = Progress(n_subs, desc="Simulating subjects...")

for sub_num in 1:n_subs
    Random.seed!(rngseed + sub_num)

    γb  = rand(γbase_dist)
    γc  = rand(γcoef_dist)
    #β   = rand(beta_dist)
    #ϵ   = rand(eps_dist)   # ← sample ϵ per subject
    alpha = rand(alpha_dist)

    # call the model (same as your working version)
    simulated_data, max_clus, total_reward = crp([alpha, γb, γc, 1])

    # add identifying parameters
    simulated_data[!, :sub_num]     .= sub_num
    simulated_data[!, :alpha]       .= alpha
    simulated_data[!, :gamma_base]  .= γb
    
    simulated_data[!, :gamma_coef]  .= γc
    simulated_data[!, :beta]        .= 1
    simulated_data[!, :epsilon]     .= 0    # ← record the lapse rate

    append!(all_data, simulated_data)
    next!(p)
end

CSV.write("simulated_structure_learning_RANDOM_300.csv", all_data)
println("✅ Finished! Saved --> simulated_structure_learning_RANDOM_300.csv")
println("Total subjects simulated: ", n_subs)



In [ ]:
# Add switch column based on galaxy transitions
function add_switch_column(data)
    switch_col = []
    for i in 1:nrow(data)
        planet = data.planet_num[i]
        curr_planet_type = data.galaxy[i]
        
        if planet > 0
            last_planet_type = data[data.planet_num .== (planet-1), :galaxy][1]
            if curr_planet_type == last_planet_type
                push!(switch_col, String("no_switch"))
            else
                push!(switch_col, String("switch"))
            end
        else
            push!(switch_col, String("no_switch"))
        end
    end
    return switch_col
end

add_switch_column (generic function with 1 method)

In [ ]:
alpha = 0.2
gamma_base = 1
gamma_coef = 0
beta = 5
# Create array to store results from each worker
p = Progress(n_subs, desc="Simulating subjects...")
results = @distributed (vcat) for sub_num in 1:n_subs
    simulated_data, max_clus, total_reward = crp([alpha, gamma_base, gamma_coef, beta])
    switch_col = add_switch_column(simulated_data)
    insertcols!(simulated_data, :switch => switch_col)
    simulated_data[!, :sub_num] .= sub_num
    next!(p)
    simulated_data
end
# Save simulated data to CSV
CSV.write("results/model_predictions/simulated_data_uncertainty_non_adapter.csv", results)


Simulating subjects... 100%|█████████████████████████████| Time: 0:10:01


"results/model_predictions/simulated_data_uncertainty_non_adapter.csv"

In [ ]:
alpha = 0.2
gamma_base = 1
gamma_coef = -0.3
beta = 5
# Create array to store results from each worker
p = Progress(n_subs, desc="Simulating subjects...")
results = @distributed (vcat) for sub_num in 1:n_subs
    simulated_data, max_clus, total_reward = crp([alpha, gamma_base, gamma_coef, beta])
    switch_col = add_switch_column(simulated_data)
    insertcols!(simulated_data, :switch => switch_col)
    simulated_data[!, :sub_num] .= sub_num
    next!(p)
    simulated_data
end
# Save simulated data to CSV
CSV.write("results/model_predictions/simulated_data_uncertainty_adapter.csv", results)


Simulating subjects... 100%|█████████████████████████████| Time: 0:09:29


"results/model_predictions/simulated_data_uncertainty_adapter.csv"

In [ ]:
using CSV
using DataFrames
using Random
using Distributions
using Statistics
using StatsBase
using ProgressMeter

include("model.jl")

# ============================================================
# Settings
# ============================================================
n_subs = 300
rngseed = 1777

gamma_base_dist = Uniform(-10.0, 10.0)
gamma_coef_dist = Uniform(-3.0, 3.0)
alpha_dist      = Uniform(0.0, 10.0)

fixed_beta    = 1.0
fixed_epsilon = 0.0

num_particles = 200
n_samples = 100
max_prt = 20

Random.seed!(rngseed)

# ============================================================
# Load exp_struc ONCE
# Important: galaxy must be indexed by block and within-block planet
# ============================================================
exp_struc = DataFrame(CSV.File("exp_struc.csv"))

required_exp_cols = [:block, :planet, :galaxy]
missing_exp_cols = setdiff(required_exp_cols, Symbol.(names(exp_struc)))

if !isempty(missing_exp_cols)
    error("exp_struc.csv is missing columns: $(missing_exp_cols). Existing columns: $(names(exp_struc))")
end

function get_galaxy_from_exp_struc(exp_struc, block_num, planet_in_block)
    rows = exp_struc[
        (exp_struc.block .== block_num) .&
        (exp_struc.planet .== planet_in_block),
        :
    ]

    if nrow(rows) == 0
        error("No galaxy found in exp_struc.csv for block=$(block_num), planet=$(planet_in_block)")
    elseif nrow(rows) > 1
        error("Multiple rows found in exp_struc.csv for block=$(block_num), planet=$(planet_in_block)")
    end

    return rows.galaxy[1]
end

function deterministic_choice(v_stay, v_leave)
    return v_stay >= v_leave ? 1 : 0
end

# ============================================================
# Simulate one subject
# ============================================================
function simulate_one_subject(
    sub_num,
    alpha,
    gamma_base,
    gamma_coef,
    exp_struc;
    num_particles=50,
    n_samples=100,
    max_prt=20
)

    expt = Experiment(5, 360, 20, 2, 10, 5.5, 1.5)

    hyper_mu = 0.5
    hyper_var = 0.25
    hyper_tau = 1.0

    particles, weights = init_particle_filter(
        num_particles,
        hyper_mu,
        hyper_var,
        hyper_tau,
        alpha
    )

    total_reward = 0.0
    total_time = expt.alien_time + expt.iti
    N = 0  # global planet counter used for CRP prior

    sim_data = DataFrame(
        sub_num = Int[],
        block = Int[],
        planet_num = Int[],
        true_planet = Int[],
        galaxy = Int[],
        stay_num = Int[],
        reward = Float64[],
        alpha = Float64[],
        gamma_base = Float64[],
        gamma_coef = Float64[],
        beta = Float64[],
        epsilon = Float64[]
    )

    for b in 1:expt.n_blocks
        block_time = expt.alien_time

        for planet_in_block in 0:(expt.block_n_planet - 1)

            if block_time >= expt.block_max_time
                break
            end

            # This is the key fix:
            # galaxy is read using within-block planet index, NOT global N
            galaxy = get_galaxy_from_exp_struc(exp_struc, b, planet_in_block)

            true_planet = (b - 1) * expt.block_n_planet + planet_in_block

            on_planet = true
            prt = 0
            curr_reward = 0.0
            decay_list = Float64[]
            prob_k_local = nothing

            while block_time < expt.block_max_time &&
                  on_planet &&
                  prt < max_prt

                # -----------------------------
                # Generate reward
                # -----------------------------
                if prt == 0
                    curr_reward = round(get_rho0())
                else
                    decay = sample_decay(galaxy)
                    curr_reward = round(curr_reward * decay)
                    push!(decay_list, decay)
                end

                total_reward += curr_reward

                # Save current harvest
                push!(sim_data, (
                    sub_num,
                    b,
                    planet_in_block,
                    true_planet,
                    galaxy,
                    prt,
                    curr_reward,
                    alpha,
                    gamma_base,
                    gamma_coef,
                    fixed_beta,
                    fixed_epsilon
                ))

                # update time
                block_time += expt.harvest_time + expt.iti
                total_time += expt.harvest_time + expt.iti

                # -----------------------------
                # CRP posterior / uncertainty
                # -----------------------------
                prob_k_local = prob_cluster(
                    curr_reward,
                    decay_list,
                    N,
                    alpha,
                    particles
                )

                samples, clusters = sample_v_stay(
                    particles,
                    weights,
                    prob_k_local,
                    n_samples,
                    decay_list
                )

                clusters = round.(Int, clusters)
                K = maximum(clusters)

                cluster_probs = tally(clusters, K) ./ n_samples
                d = Multinomial(n_samples, cluster_probs)
                model_uncertainty = entropy(d)

                gamma_effective = 1 / (
                    1 + exp(-(gamma_base - gamma_coef * model_uncertainty))
                )

                v_stay = mean(samples) * curr_reward

                v_leave = (
                    total_reward / total_time
                ) * (expt.harvest_time + expt.iti) * gamma_effective

                # deterministic policy
                action = deterministic_choice(v_stay, v_leave)

                prt += 1
                on_planet = action == 1
            end

            # update particle filter after leaving planet
            if length(decay_list) > 0 && prob_k_local !== nothing
                particles, weights = resample_and_update_particles(
                    particles,
                    weights,
                    prob_k_local,
                    decay_list
                )
            end

            # travel to next planet
            block_time += expt.travel_time + expt.alien_time
            total_time += expt.travel_time + expt.alien_time

            N += 1
        end
    end

    return sim_data
end

# ============================================================
# Simulate all subjects
# ============================================================
all_data = DataFrame()
p = Progress(n_subs, desc="Simulating subjects...")

for sub_num in 1:n_subs
    Random.seed!(rngseed + sub_num)

    alpha = rand(alpha_dist)
    gamma_base = rand(gamma_base_dist)
    gamma_coef = rand(gamma_coef_dist)

    sub_data = simulate_one_subject(
        sub_num,
        alpha,
        gamma_base,
        gamma_coef,
        exp_struc;
        num_particles=num_particles,
        n_samples=n_samples,
        max_prt=max_prt
    )

    append!(all_data, sub_data)
    next!(p)
end

out_file = "simulated_structure_learning_DETERMINISTIC_300.csv"
CSV.write(out_file, all_data)

println("✅ Finished! Saved --> ", out_file)
println("Total subjects simulated: ", n_subs)
println("Total rows: ", nrow(all_data))
println("Maximum stay_num: ", maximum(all_data.stay_num))

Simulating subjects... 100%|█████████████████████████████| Time: 0:18:13


✅ Finished! Saved --> simulated_structure_learning_DETERMINISTIC_300.csv
Total subjects simulated: 300
Total rows: 88795
Maximum stay_num: 19
